# Test: Splice Usage Head Training Verification

This notebook verifies that the splice usage head is receiving gradients and training correctly.

**What we're testing:**
1. Usage head receives non-zero gradients
2. Predictions change after training steps
3. Valid (position, condition) pairs are being matched
4. Loss is non-zero

In [1]:
import sys
import torch
import numpy as np
from pathlib import Path

# Add project root to path if needed
project_root = Path("/home/elek/projects/alphagenome_ft_pytorch")
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

PyTorch version: 2.10.0+cu128
CUDA available: True
Device: cuda


## 1. Load Model and Create Usage Head

In [2]:
from alphagenome_pytorch import AlphaGenome
from alphagenome_pytorch.config import DtypePolicy
from alphagenome_pytorch.extensions.finetuning.heads import create_splice_usage_finetuning_head
from alphagenome_pytorch.extensions.finetuning.transfer import load_trunk, remove_all_heads

# Load pretrained model
pretrained_path = project_root / "checkpoints" / "model_fold_0.safetensors"
print(f"Loading model from: {pretrained_path}")

# Use full float32 for simplicity in testing (mixed_precision requires dtype conversion)
dtype_policy = DtypePolicy.default()
model = AlphaGenome(dtype_policy=dtype_policy)
model = load_trunk(model, str(pretrained_path), exclude_heads=True)
model = remove_all_heads(model)

# Create usage head
n_conditions = 86
usage_head = create_splice_usage_finetuning_head(n_conditions=n_conditions, num_organisms=1)

model = model.to(device).eval()
usage_head = usage_head.to(device)

print(f"Model loaded. Usage head: {n_conditions} conditions")
print(f"Usage head params: {sum(p.numel() for p in usage_head.parameters()):,}")

Loading model from: /home/elek/projects/alphagenome_ft_pytorch/checkpoints/model_fold_0.safetensors
Model loaded. Usage head: 86 conditions
Usage head params: 132,182


## 2. Load Small Dataset Sample

Use a few batches from the real dataset to test coordinate matching.

In [3]:
from alphagenome_pytorch.extensions.finetuning.splice_datasets import (
    SpliceSiteAnnotation,
    SpliceSiteUsageIndex,
    SpliceSiteDataset,
    collate_splice,
)
from alphagenome_pytorch.extensions.finetuning.datasets import CachedGenome
from torch.utils.data import DataLoader

# Paths (adjust to your data)
genome_path = "/home/elek/sds/sd17d003/Anamaria/genomes/ensembl115/fasta/Homo_sapiens.fa"
annotation_path = "/home/elek/sds/sd17d003/Anamaria/alphagenome_genomicsxai/data/Homo_sapiens/splice_sites.parquet"
usage_path = "/home/elek/sds/sd17d003/Anamaria/alphagenome_genomicsxai/data/Homo_sapiens/usage.parquet"
train_bed = "/home/elek/sds/sd17d003/Anamaria/alphagenome_genomicsxai/data/Homo_sapiens/folds_100kb/FOLD_0_subset/train.bed"

print("Loading annotation...")
annotation = SpliceSiteAnnotation(annotation_path)

coord_base = 0
print(f"usage_coord_base={coord_base}:")
usage_index = SpliceSiteUsageIndex(usage_path, usage_coord_base=coord_base)

# Create minimal dataset (just a few samples)
dataset = SpliceSiteDataset(
    genome=genome_path,
    bed_file=train_bed,
    annotation=annotation,
    usage_index=usage_index,
    sequence_length=131072,
    organism_index=0,
    max_sites=1024,
)


Loading annotation...
SpliceSiteAnnotation: loaded 354,344 sites across 56 chromosomes from splice_sites.parquet
usage_coord_base=0:
SpliceSiteUsageIndex: loaded 714,350 sites, 86 conditions from usage.parquet
CachedGenome: Loading genome from /home/elek/sds/sd17d003/Anamaria/genomes/ensembl115/fasta/Homo_sapiens.fa...
CachedGenome: Loaded 1 chromosomes (995.8 MB)


In [4]:
# Check a sample for valid usage pairs
sample = dataset[0]
if "usage_mask" in sample:
    n_valid = sample["usage_mask"].sum().item()
    print(f"Sample 0: {n_valid} valid (position, condition) pairs")
else:
    print(f"Sample 0: No usage data")

Sample 0: 2838 valid (position, condition) pairs


In [5]:
# Use the correct coordinate base
usage_index = SpliceSiteUsageIndex(usage_path, usage_coord_base=0)
dataset = SpliceSiteDataset(
    genome=genome_path,
    bed_file=train_bed,
    annotation=annotation,
    usage_index=usage_index,
    sequence_length=131072,
    organism_index=0,
    max_sites=1024,
)

print(f"\nDataset: {len(dataset)} samples")

SpliceSiteUsageIndex: loaded 714,350 sites, 86 conditions from usage.parquet
CachedGenome: Loading genome from /home/elek/sds/sd17d003/Anamaria/genomes/ensembl115/fasta/Homo_sapiens.fa...
CachedGenome: Loaded 1 chromosomes (995.8 MB)

Dataset: 1000 samples


In [6]:
# Create dataloader with just a few samples for quick test
loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_splice,
)

print(f"Loader: {len(loader)} batches")

Loader: 500 batches


In [8]:
# Get one batch and check for valid usage data
batch = next(iter(loader))
print(f"Usage positions shape: {batch['usage_positions'].shape}")
batch['usage_positions']

Usage positions shape: torch.Size([2, 1024])


tensor([[   53,   111,  2272,  ...,    -1,    -1,    -1],
        [17745, 18385, 50849,  ...,    -1,    -1,    -1]])

In [9]:
print(f"Usage mask shape: {batch['usage_mask'].shape}")
batch['usage_mask']

Usage mask shape: torch.Size([2, 1024, 86])


tensor([[[ True,  True,  True,  ...,  True,  True,  True],
         [ True,  True,  True,  ...,  True,  True,  True],
         [ True,  True,  True,  ...,  True,  True,  True],
         ...,
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False]],

        [[ True,  True,  True,  ...,  True,  True,  True],
         [ True,  True,  True,  ...,  True,  True,  True],
         [ True,  True,  True,  ...,  True,  True,  True],
         ...,
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False],
         [False, False, False,  ..., False, False, False]]])

In [10]:
print(f"Usage values shape: {batch['usage_values'].shape}")
batch['usage_values']

Usage values shape: torch.Size([2, 1024, 86])


tensor([[[0.9480, 0.9790, 0.9520,  ..., 0.9790, 0.9460, 0.9780],
         [0.9340, 0.9470, 0.8520,  ..., 0.9660, 0.8240, 0.9500],
         [0.6690, 0.4290, 0.7420,  ..., 0.6880, 0.2190, 0.5820],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],

        [[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.7500, 0.6250, 0.6670,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]]])

## 3. Test Forward Pass and Check Valid Pairs

In [11]:
# Get one batch and check for valid usage data
batch = next(iter(loader))

print("Batch contents:")
for k, v in batch.items():
    if torch.is_tensor(v):
        print(f"  {k}: {v.shape}")

if "usage_mask" in batch:
    n_valid = batch["usage_mask"].sum().item()
    print(f"\nValid (position, condition) pairs in batch: {n_valid}")
    if n_valid == 0:
        print("\nNo valid pairs! Usage head won't receive gradients.")
else:
    print("\nNo usage data in batch!")

Batch contents:
  sequence: torch.Size([2, 131072, 4])
  organism_index: torch.Size([2])
  classification_labels: torch.Size([2, 131072])
  usage_positions: torch.Size([2, 1024])
  usage_values: torch.Size([2, 1024, 86])
  usage_mask: torch.Size([2, 1024, 86])

Valid (position, condition) pairs in batch: 11094


## 4. Run Training Step and Check Gradients

In [ ]:
from alphagenome_pytorch.extensions.finetuning.splice_losses import splice_usage_loss
import torch.nn.functional as F

# Freeze model, train only usage head
for p in model.parameters():
    p.requires_grad = False
usage_head.train()

optimizer = torch.optim.AdamW(usage_head.parameters(), lr=1e-4)

print("Running training step...\n")

# Get predictions before training
with torch.no_grad():
    seq = batch["sequence"].to(device)
    org_idx = batch["organism_index"].to(device)
    
    outputs = model.forward(seq, org_idx, resolutions=(1,), channels_last=False, embeddings_only=True)
    emb_1bp = outputs["embeddings_1bp"]
    
    usage_out_before = usage_head(emb_1bp, org_idx, channels_last=True)
    preds_before = torch.sigmoid(usage_out_before["logits"]).cpu().numpy()
    print(f"Predictions BEFORE training:")
    print(f"  Mean: {preds_before.mean():.4f}, Std: {preds_before.std():.4f}")
    print(f"  Min: {preds_before.min():.4f}, Max: {preds_before.max():.4f}")

# Training step
n_steps = 10
losses = []
n_valid_list = []

for step in range(n_steps):
    optimizer.zero_grad()
    
    outputs = model.forward(seq, org_idx, resolutions=(1,), channels_last=False, embeddings_only=True)
    emb_1bp = outputs["embeddings_1bp"]
    
    usage_out = usage_head(emb_1bp, org_idx, channels_last=True)
    
    if "usage_positions" in batch:
        usage_pos = batch["usage_positions"].to(device)
        usage_vals = batch["usage_values"].to(device)
        usage_mask = batch["usage_mask"].to(device)
        
        loss, metrics = splice_usage_loss(
            usage_out["logits"], usage_pos, usage_vals, usage_mask
        )
        
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        n_valid_list.append(metrics["n_valid"])
        
        if step == 0:
            print(f"\nStep {step+1}:")
            print(f"  Loss: {loss.item():.4f}")
            print(f"  Valid pairs: {metrics['n_valid']}")
            print(f"  Correlation: {metrics.get('correlation', 'N/A')}")
            
            # Check gradients
            grad_norms = []
            for name, param in usage_head.named_parameters():
                if param.grad is not None:
                    grad_norm = param.grad.norm().item()
                    grad_norms.append(grad_norm)
                    if grad_norm > 0:
                        print(f"  ✓ {name}: grad_norm={grad_norm:.6f}")
            
            if all(g == 0 for g in grad_norms):
                print("\n  All gradients are zero! Usage head won't train.")
            else:
                print(f"\n  Non-zero gradients detected!")

print(f"\n{'='*60}")
print(f"Completed {n_steps} training steps")
print(f"  Average loss: {np.mean(losses):.4f}")
print(f"  Average valid pairs: {np.mean(n_valid_list):.1f}")

if np.mean(n_valid_list) == 0:
    print("\nNo valid pairs found!")
    print("   → Coordinate mismatch likely. Try usage_coord_base=0 instead.")
else:
    print(f"\n✓ Usage head is receiving gradients!")

Running training step...

Predictions BEFORE training:
  Mean: 0.5125, Std: 0.1358
  Min: 0.0015, Max: 0.9979

Step 1:
  Loss: 0.7845
  Valid pairs: 11094
  Correlation: -0.087419293820858
  ✓ conv.weight: grad_norm=0.545927
  ✓ conv.bias: grad_norm=0.022357

  Non-zero gradients detected!

Completed 10 training steps
  Average loss: 0.7431
  Average valid pairs: 11094.0

✓ Usage head is receiving gradients!


## 5. Check Predictions Changed

In [13]:
# Get predictions after training
with torch.no_grad():
    usage_head.eval()
    outputs = model.forward(seq, org_idx, resolutions=(1,), channels_last=False, embeddings_only=True)
    emb_1bp = outputs["embeddings_1bp"]
    usage_out_after = usage_head(emb_1bp, org_idx, channels_last=True)
    preds_after = torch.sigmoid(usage_out_after["logits"]).cpu().numpy()

print(f"Predictions AFTER {n_steps} training steps:")
print(f"  Mean: {preds_after.mean():.4f}, Std: {preds_after.std():.4f}")
print(f"  Min: {preds_after.min():.4f}, Max: {preds_after.max():.4f}")

# Check if predictions changed
diff = np.abs(preds_after - preds_before)
print(f"\nChange in predictions:")
print(f"  Mean abs change: {diff.mean():.6f}")
print(f"  Max abs change: {diff.max():.6f}")

if diff.mean() < 1e-6:
    print("\n⚠️  Predictions did NOT change! Usage head is not training.")
    print("   Check coordinate base and valid pair count.")
else:
    print(f"\n✓ Predictions changed! Usage head is training correctly.")

Predictions AFTER 10 training steps:
  Mean: 0.5066, Std: 0.1312
  Min: 0.0022, Max: 0.9975

Change in predictions:
  Mean abs change: 0.036038
  Max abs change: 0.180859

✓ Predictions changed! Usage head is training correctly.


## 6. Summary

**Expected results if training correctly:**
- ✓ Valid (position, condition) pairs > 0
- ✓ Non-zero gradients in usage head parameters
- ✓ Loss > 0 and decreasing
- ✓ Predictions change after training steps

**If any checks fail:**
1. Verify `usage_coord_base` matches your data (run scripts/diagnose_coordinate_mismatch.py)
2. Check chromosome naming consistency between annotation and usage parquet
3. Ensure usage parquet was generated from the same genome version as annotation

In [ ]:
# Final diagnostic summary
print("="*60)
print("DIAGNOSTIC SUMMARY")
print("="*60)

checks = [
    ("Valid pairs found", np.mean(n_valid_list) > 0),
    ("Loss is non-zero", np.mean(losses) > 0),
    ("Predictions changed", diff.mean() > 1e-6),
]

all_pass = True
for check_name, passed in checks:
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"{status}: {check_name}")
    if not passed:
        all_pass = False

print("="*60)
if all_pass:
    print("\nAll checks passed! Usage head is training correctly.")
else:
    print("\nSome checks failed. Review coordinate base setting.")
    print("   Run scripts/diagnose_coordinate_mismatch.py to verify your data.")

DIAGNOSTIC SUMMARY
✓ PASS: Valid pairs found
✓ PASS: Loss is non-zero
✓ PASS: Predictions changed

All checks passed! Usage head is training correctly.
